# ReWear AI — Análisis exploratorio de datasets

Notebook reproducible para inspeccionar imágenes y anotaciones **sin modificar los originales**. No entrena modelos. Resume CISUTAC Holes/Spots, FabricSpotDefect, FabriD v3, RAW-FABRID y la muestra autorizada de RISE.

In [ ]:
from pathlib import Path
import json, os, sys
import matplotlib.pyplot as plt
from PIL import Image
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from eda_utils import image_files, image_inventory, analyze_raw_fabrid, analyze_yolo_segmentation
DATA = Path(os.environ.get('REWEAR_DATA_DIR', ROOT / 'data')).expanduser().resolve()
print('Raíz:', ROOT, '| Datos:', DATA)

## Inventario y resoluciones
Se cuentan archivos y se leen únicamente sus cabeceras. La heterogeneidad de resolución es una señal del cambio de dominio entre prendas usadas e inspección industrial.

In [ ]:
roots = {
 'CISUTAC Holes': DATA/'cisutac_holes', 'CISUTAC Spots': DATA/'cisutac_spots',
 'FabriD original': DATA/'fabrid_v3'/'original',
 'RAW-FABRID high-res': DATA/'raw_fabrid'/'extracted'/'RAW_FABRID'/'images',
 'RISE muestra': DATA/'rise_samples'}
for name, path in roots.items():
    inv = image_inventory(path)
    print(f'\n{name}: {inv["images"]} imágenes | modos={inv["modes"]}')
    print('Resoluciones más frecuentes:', sorted(inv['resolutions'].items(), key=lambda x:-x[1])[:6])

## Ejemplos visuales
Se muestran muestras de cada dominio. Para RISE sólo existen tres imágenes descargadas deliberadamente; el corpus completo no se descargó.

In [ ]:
fig, axes = plt.subplots(len(roots), 3, figsize=(12, 4*len(roots)))
for row, (name, path) in enumerate(roots.items()):
    files = image_files(path)[:3]
    for col, ax in enumerate(axes[row]):
        ax.axis('off'); ax.set_title(name)
        if col < len(files): ax.imshow(Image.open(files[col])); ax.set_title(f'{name}\n{files[col].name[:35]}')
plt.tight_layout()

## FabriD v3: clases, particiones y polígonos
Los archivos vacíos corresponden a imágenes defect-free. La comprobación por nombre fuente detecta capturas emparentadas presentes en más de una partición.

In [ ]:
fabrid = analyze_yolo_segmentation(DATA/'fabrid_v3'/'annotations_export', ['Needle Mark Defect','hole Defect','missing yarn Defect','stain Defect'])
display(fabrid['splits'], fabrid['instances'], fabrid['area_fraction'])
print('Fuentes posiblemente filtradas entre splits:', len(fabrid['possible_source_leakage']))

## RAW-FABRID: aptitud para anomalías y trazabilidad
Los parches conservan `parent_raw`, lo que permite dividir por captura o rollo y evitar que parches vecinos de la misma tela aparezcan a ambos lados de una evaluación.

In [ ]:
raw = analyze_raw_fabrid(DATA/'raw_fabrid'/'extracted'/'RAW_FABRID')
display(raw)

## Lectura experimental recomendada

- **Supervisado en prendas usadas:** CISUTAC Holes/Spots como tareas separadas; no poseen normales ni test oficial.
- **Localización industrial:** FabriD v3 (cuatro clases, polígonos) y FabricSpotDefect (boxes y algunas segmentaciones).
- **PatchCore/anomalías:** RAW-FABRID es el benchmark más limpio y directamente compatible; FabriD defect-free puede servir como segundo dominio.
- **Prendas normales reales:** RISE podría aportar negativos débiles, pero `stains/holes` tienen faltantes históricos y no hay máscaras ni cajas. Debe filtrarse conservadoramente y auditarse visualmente.
- **Evaluación:** reportar resultados por dataset; no mezclar dominios industriales y prendas usadas como si fueran i.i.d.